In [1]:
import networkx as graphs
from itertools import combinations
from collections import Counter
import pandas as pd
import requests
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS as stopwords

In [2]:
WIKI_API = "https://en.wikipedia.org/w/api.php"
HEADERS = {
    "User-Agent": "Python/requests"
}

In [3]:
def wiki(title):
    params = {
        "action": "query",
        "prop": "extracts|info",
        "explaintext": False,
        "titles": title,
        "inprop" : "url",
        "format": "json",
        "redirects": 0,
        "formatversion": 2,
        "exintro" : 1
    }
    response = requests.get(WIKI_API, params=params, headers=HEADERS)
    pages = response.json().get("query", {}).get("pages", [])
    if not pages:
        return ""
    return pages[0].get("extract", "") ,pages[0].get("canonicalurl",""), pages[0].get("title","")

<h4>all of the snippets above were fetched from previous assignments and slightly modified to comply with the requirements of this assignment</h4>

In [4]:
def buildgraph(text):
    tokens =[t for t in (re.split(r"\W+", text.lower())) if t.isalpha() and t not in stopwords and len(t)>1]
    graph = graphs.DiGraph() #  coocurences are considered as one directional (ltr), hence the use of a directed graph
    for i in range(len(tokens)-1):
        if tokens[i]!=tokens[i+1]:
            if graph.has_edge(tokens[i],tokens[i+1]):
                graph[tokens[i]][tokens[i+1]]["weight"]+=1
            else:
                graph.add_edge(tokens[i],tokens[i+1],weight=1)
    return graph

<h4>The core logic gor this assignment: we loop through the text and increment the weights in a directed graph according to the co-occurences in it</h4>

In [5]:
def scoregraph(graph):
    scores = graphs.pagerank(graph, weight = 'weight') #since TextRank is essentialy a special case of PageRank, we can apply networkx's built-in PageRank and consider the assigned weights
    rankings = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return rankings

<h4>As explained in the code itself, we apply the built-in PageRank to score the created graph</h4>

In [6]:
text, url,title = wiki('chocolate')
df = pd.DataFrame(scoregraph(buildgraph(text))[:5],columns=["keyword", "TextRank score"])
print(f"page title: {title}, url: {url}")
df

page title: Chocolate, url: https://en.wikipedia.org/wiki/Chocolate


,keyword,TextRank score
0,chocolate,0.082135
1,cocoa,0.043755
2,cacao,0.017551
3,food,0.016858
4,consumed,0.014862
